# Qwen2.5-Omni vLLM Server (GPTQ-Int4)

Tests vLLM open-source inference with Qwen Omni model.
Validates OpenAI-compatible API for text + audio I/O.

| Item | Value |
|------|-------|
| Model | `Qwen/Qwen2.5-Omni-7B-GPTQ-Int4` |
| VRAM | ~5GB (T4 16GB OK) |
| API | OpenAI-compatible `/v1/chat/completions` |
| Backend | vLLM (standard, VLLM_USE_V1=0) |
| Purpose | vLLM API compatibility test for naia-os |

## 1. GPU Check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU! Change runtime: Runtime > Change runtime type > T4 GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nGPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")

## 2. Install vLLM-Omni

In [ ]:
%%time
!pip install -q vllm
!pip install -q "protobuf>=5.26,<6"
!pip install -q qwen-omni-utils openai

import vllm; print(f"vllm: {vllm.__version__}")
import google.protobuf; print(f"protobuf: {google.protobuf.__version__}")

## 3. Launch vLLM-Omni Server

In [ ]:
import subprocess, os

MODEL = "Qwen/Qwen2.5-Omni-7B-GPTQ-Int4"
PORT = 8091

log_file = open("/tmp/vllm-server.log", "w")

env = {
    **os.environ,
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "VLLM_LOGGING_LEVEL": "INFO",
}
# Remove VLLM_USE_V1 — let it use V1 (default in 0.17.0)
env.pop("VLLM_USE_V1", None)

server_proc = subprocess.Popen(
    [
        "vllm", "serve", MODEL,
        "--port", str(PORT),
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.90",
        "--max-num-seqs", "1",
        "--trust-remote-code",
        "--quantization", "gptq",
        "--enforce-eager",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)

print(f"Server PID: {server_proc.pid}")
print(f"Model: {MODEL}")
print("V1 engine + enforce-eager")
print("Loading... (3-5 min)")
print("Check logs: !cat /tmp/vllm-server.log")

## 4. Wait for Server Ready

In [ ]:
import urllib.request, time

url = f"http://localhost:{PORT}/health"
print("Waiting for server...")

for i in range(120):
    try:
        r = urllib.request.urlopen(url, timeout=2)
        if r.status == 200:
            print(f"\n\nServer ready! (took {(i+1)*5}s)")
            break
    except Exception:
        pass
    if server_proc.poll() is not None:
        print("\n\nServer crashed! Logs:")
        !tail -50 /tmp/vllm-server.log
        break
    print(".", end="", flush=True)
    time.sleep(5)
else:
    print("\n\nTimeout. Logs:")
    !tail -30 /tmp/vllm-server.log

## 5. Tunnel (for naia-os remote access)

Copy the tunnel URL → use as `baseURL` in naia-os.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, re, time

tunnel_log = open("/tmp/tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

time.sleep(10)

with open("/tmp/tunnel.log") as f:
    log = f.read()
urls = re.findall(r"https://[\w-]+\.trycloudflare\.com", log)
if urls:
    TUNNEL_URL = urls[0]
    print(f"{'='*60}")
    print(f"  TUNNEL URL: {TUNNEL_URL}")
    print(f"  API:        {TUNNEL_URL}/v1/chat/completions")
    print(f"{'='*60}")
else:
    print("Tunnel URL not found yet. Re-run or check:")
    !cat /tmp/tunnel.log

## 6. Test: Text Only

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="EMPTY")

print("--- Text Only ---")
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello! Say hi in Korean. One sentence."}],
    modalities=["text"],
)
print(f"Response: {resp.choices[0].message.content}")
print(f"Usage: input={resp.usage.prompt_tokens}, output={resp.usage.completion_tokens}")

## 7. Test: Audio Output (Text → Speech)

In [ ]:
import base64, json
from IPython.display import Audio, display

print("--- Audio Output ---")
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in Korean."}],
    modalities=["text", "audio"],
)

for i, choice in enumerate(resp.choices):
    print(f"choice[{i}]:")
    msg = choice.message
    if msg.content:
        print(f"  text: {msg.content}")
    # Audio may be in message.audio (dict or str)
    audio = getattr(msg, 'audio', None)
    if audio:
        audio_data = audio.get('data', audio) if isinstance(audio, dict) else audio
        if isinstance(audio_data, str) and len(audio_data) > 100:
            wav_bytes = base64.b64decode(audio_data)
            with open(f"/tmp/vllm_tts_{i}.wav", "wb") as f:
                f.write(wav_bytes)
            print(f"  audio: {len(wav_bytes)} bytes → /tmp/vllm_tts_{i}.wav")
            display(Audio(f"/tmp/vllm_tts_{i}.wav"))

## 8. Test: Audio Input (Speech → Text)

In [ ]:
import numpy as np, struct, io

# Generate 1s test WAV (440Hz sine)
sr = 16000
t = np.linspace(0, 1.0, sr, endpoint=False)
samples = (np.sin(2 * np.pi * 440 * t) * 32767).astype(np.int16)
buf = io.BytesIO()
n = len(samples)
buf.write(b"RIFF")
buf.write(struct.pack("<I", 36 + n*2))
buf.write(b"WAVEfmt ")
buf.write(struct.pack("<IHHIIHH", 16, 1, 1, sr, sr*2, 2, 16))
buf.write(b"data")
buf.write(struct.pack("<I", n*2))
buf.write(samples.tobytes())
audio_b64 = base64.b64encode(buf.getvalue()).decode()

print("--- Audio Input → Text ---")
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "audio_url", "audio_url": {"url": f"data:audio/wav;base64,{audio_b64}"}},
            {"type": "text", "text": "What do you hear? Describe this audio."},
        ],
    }],
    modalities=["text"],
)
print(f"Response: {resp.choices[0].message.content}")

## 9. Test: Audio In → Audio Out

In [ ]:
print("--- Audio In → Audio Out ---")
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "audio_url", "audio_url": {"url": f"data:audio/wav;base64,{audio_b64}"}},
            {"type": "text", "text": "Repeat what you heard and describe it."},
        ],
    }],
    modalities=["text", "audio"],
)

for i, choice in enumerate(resp.choices):
    print(f"choice[{i}]:")
    msg = choice.message
    if msg.content:
        print(f"  text: {msg.content}")
    audio = getattr(msg, 'audio', None)
    if audio:
        audio_data = audio.get('data', audio) if isinstance(audio, dict) else audio
        if isinstance(audio_data, str) and len(audio_data) > 100:
            wav_bytes = base64.b64decode(audio_data)
            with open(f"/tmp/vllm_omni_{i}.wav", "wb") as f:
                f.write(wav_bytes)
            print(f"  audio: {len(wav_bytes)} bytes")
            display(Audio(f"/tmp/vllm_omni_{i}.wav"))

## 10. API Format Summary

Print the actual request/response format for naia-os integration.

In [ ]:
import json

print("=" * 60)
print("vLLM-Omni API Format Summary (for naia-os provider)")
print("=" * 60)
print()
print("Endpoint: POST /v1/chat/completions")
print(f"Model: {MODEL}")
print()
print("--- Text Only ---")
print(json.dumps({
    "model": MODEL,
    "messages": [{"role": "user", "content": "..."}],
    "modalities": ["text"],
}, indent=2))
print()
print("--- Text + Audio Output ---")
print(json.dumps({
    "model": MODEL,
    "messages": [{"role": "user", "content": "..."}],
    "modalities": ["text", "audio"],
}, indent=2))
print()
print("--- Audio Input ---")
print(json.dumps({
    "model": MODEL,
    "messages": [{"role": "user", "content": [
        {"type": "audio_url", "audio_url": {"url": "data:audio/wav;base64,..."}},
        {"type": "text", "text": "..."},
    ]}],
    "modalities": ["text"],
}, indent=2))

## 11. Logs (Debug)

In [ ]:
!tail -30 /tmp/vllm-server.log

## 12. Cleanup

In [ ]:
server_proc.terminate()
tunnel_proc.terminate()
print("Done.")